# database check

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd()
DB_PATH = PROJECT_ROOT / "data" / "institutional_holding.db"

connection = sqlite3.connect(DB_PATH)
print(f"数据库位置：{DB_PATH}")
print(f"数据库存在：{DB_PATH.exists()}")

数据库位置：d:\KimiData\kimi\workspace\institutional_holding_tracker\data\institutional_holding.db
数据库存在：True


## 查看数据库表

In [12]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name
    """,
    connection,
)

tables

,name
0,alerts
1,daily_prices
2,fund_holdings
3,holder_mappings
4,holding_changes_summary
5,index_components
6,index_holding_summary
7,indices
8,northbound_holdings
9,sqlite_sequence


## 查看指数基本信息

In [17]:
indices = pd.read_sql_query(
    "SELECT * FROM indices ORDER BY index_code",
    connection,
)

indices

,id,index_name,index_code,exchange,component_count,updated_at
0,13,沪深300,000300,sh,300,2026-08-19 10:34:31
1,16,科创50,000688,sh,50,2026-08-19 10:34:31
2,14,中证500,000905,sh,500,2026-08-19 10:34:31
3,15,创业板指,399006,sz,100,2026-08-19 10:34:31


## 查看指数成分股

In [5]:
components = pd.read_sql_query(
    """
    SELECT index_code, stock_code, stock_name, weight, effective_date
    FROM index_components
    ORDER BY index_code, stock_code
    LIMIT 20
    """,
    connection,
)

components

,index_code,stock_code,stock_name,weight,effective_date
0,000300,000001,平安银行,0.433,2026-08-18
1,000300,000002,万科A,0.087,2026-08-18
2,000300,000063,中兴通讯,0.418,2026-08-18
3,000300,000100,TCL科技,0.378,2026-08-18
4,000300,000157,中联重科,0.145,2026-08-18
5,000300,000166,申万宏源,0.162,2026-08-18
6,000300,000301,东方盛虹,0.119,2026-08-18
7,000300,000333,美的集团,1.634,2026-08-18
8,000300,000338,潍柴动力,0.577,2026-08-18
9,000300,000408,藏格矿业,0.243,2026-08-18


In [14]:
component_counts = pd.read_sql_query(
    """
    SELECT index_code, COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code
    ORDER BY index_code
    """,
    connection,
)

component_counts

,index_code,stock_count
0,000300,300


## 统计所有指数并随机查看成分股

In [18]:
all_components = pd.read_sql_query(
    """
    SELECT index_code, stock_code, stock_name, weight, effective_date
    FROM index_components
    ORDER BY index_code, stock_code
    """,
    connection,
)

index_summary = (
    all_components.groupby("index_code", as_index=False)
    .agg(stock_count=("stock_code", "nunique"))
    .sort_values("index_code")
)

print(f"实际写入的指数数量：{len(index_summary)}")
index_summary

实际写入的指数数量：4


,index_code,stock_count
0,000300,300
1,000688,50
2,000905,500
3,399006,100


In [7]:
random_samples = pd.concat(
    [
        group.sample(n=min(5, len(group)), random_state=42)
        for _, group in all_components.groupby("index_code")
    ],
    ignore_index=True,
).sort_values(["index_code", "stock_code"])

random_samples

,index_code,stock_code,stock_name,weight,effective_date
3,000300,000408,藏格矿业,0.243,2026-08-18
2,000300,600372,中航机载,0.105,2026-08-18
0,000300,601077,渝农商行,0.138,2026-08-18
4,000300,601633,长城汽车,0.080,2026-08-18
1,000300,603019,中科曙光,0.469,2026-08-18


## 诊断指数基本信息写入

In [8]:
from config.settings import TRACKED_INDICES
from database.db_manager import query_sql
from ingestion.index_components import _update_indices_info

print("配置中的指数：")
for name, info in TRACKED_INDICES.items():
    print(name, info)

_update_indices_info()

updated_indices = pd.DataFrame(query_sql(
    "SELECT * FROM indices ORDER BY index_code"
))
updated_indices

配置中的指数：
沪深300 {'code': '000300', 'exchange': 'sh'}
中证500 {'code': '000905', 'exchange': 'sh'}
创业板指 {'code': '399006', 'exchange': 'sz'}
科创50 {'code': '000688', 'exchange': 'sh'}


,id,index_name,index_code,exchange,component_count,updated_at
0,1,沪深300,000300,sh,300,2026-08-19 09:29:39
1,4,科创50,000688,sh,0,2026-08-19 09:29:39
2,2,中证500,000905,sh,0,2026-08-19 09:29:39
3,3,创业板指,399006,sz,0,2026-08-19 09:29:39


## 回滚本次诊断写入

In [10]:
index_codes = tuple(info["code"] for info in TRACKED_INDICES.values())
placeholders = ", ".join("?" for _ in index_codes)

connection.execute(
    f"DELETE FROM indices WHERE index_code IN ({placeholders})",
    index_codes,
)
connection.commit()

remaining_indices = pd.read_sql_query(
    "SELECT * FROM indices ORDER BY index_code",
    connection,
)
remaining_components = pd.read_sql_query(
    """
    SELECT index_code, COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code
    ORDER BY index_code
    """,
    connection,
)

print(f"回滚后 indices 记录数：{len(remaining_indices)}")
print("成分股明细统计：")
remaining_components

回滚后 indices 记录数：0
成分股明细统计：


,index_code,stock_count
0,000300,300


# 小样本采集测试

## 数据表

In [22]:
table_check = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table' AND name = 'top_holders'
    """,
    connection,
)

table_check

,name
0,top_holders


## 单只股票测试接口

In [19]:
from ingestion.top_holders import fetch_top_holders_em

df = fetch_top_holders_em("000001", "20250331", tRUE)

print("shape:", df.shape)
print("columns:", list(df.columns))
display(df.head(10))

shape: (10, 7)
columns: ['名次', '股东名称', '股份类型', '持股数', '占总股本持股比例', '增减', '变动比率']


,名次,股东名称,股份类型,持股数,占总股本持股比例,增减,变动比率
0,1,中国平安保险(集团)股份有限公司-集团本级-自有资金,流通A股,9618540236,49.56,不变,NaN
1,2,中国平安人寿保险股份有限公司-自有资金,流通A股,1186100488,6.11,不变,NaN
2,3,香港中央结算有限公司,流通A股,658114653,3.39,-88767070,-11.885024
3,4,中国平安人寿保险股份有限公司-传统-普通保险产品,流通A股,440478714,2.27,不变,NaN
4,5,中国证券金融股份有限公司,流通A股,429232688,2.21,不变,NaN
5,6,中国工商银行股份有限公司-华泰柏瑞沪深300交易型开放式指数证券投资基金,流通A股,158803503,0.82,-8714000,-5.201844
6,7,中国建设银行股份有限公司-易方达沪深300交易型开放式指数发起式证券投资基金,流通A股,110935144,0.57,-4615700,-3.994519
7,8,中国工商银行股份有限公司-华夏沪深300交易型开放式指数证券投资基金,流通A股,75280777,0.39,-1530300,-1.992291
8,9,中国银行股份有限公司-嘉实沪深300交易型开放式指数证券投资基金,流通A股,70005962,0.36,-2766300,-3.801311
9,10,深圳中电投资有限公司,流通A股,62523366,0.32,不变,NaN


In [21]:
from ingestion.top_holders import ingest_all_top_holders

ingest_all_top_holders(
    ["000001"],
    ["20250331", "20241231"],
)

2026-08-19 19:52:41 [INFO] ingestion.top_holders: [TopHolders] Start ingesting 十大股东 for 1 stocks x 2 dates...
2026-08-19 19:52:43 [INFO] ingestion.top_holders: [TopHolders] Ingestion completed. Total records: 20
2026-08-19 19:52:43 [INFO] ingestion.top_holders: [TopHolders] Start ingesting 十大流通股东 for 1 stocks x 2 dates...
2026-08-19 19:52:44 [INFO] ingestion.top_holders: [TopHolders] Ingestion completed. Total records: 20


In [24]:
# 获取沪深300（000300）300只成分股的十大股东数据
top_holders_schema = pd.read_sql_query(
    "PRAGMA table_info(top_holders)",
    connection,
)

display(top_holders_schema)

top_holders_50 = pd.read_sql_query(
    """
    SELECT th.*
    FROM top_holders AS th
    INNER JOIN (
        SELECT DISTINCT stock_code
        FROM index_components
        WHERE index_code = '000300'
    ) AS c
        ON c.stock_code = th.stock_code
    ORDER BY th.stock_code
    """,
    connection,
)

print(f"获取记录数：{len(top_holders_50)}")
print(f"覆盖股票数：{top_holders_50['stock_code'].nunique()}")

display(top_holders_50)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,NaN,1
1,1,stock_code,TEXT,1,NaN,0
2,2,stock_name,TEXT,0,NaN,0
3,3,report_date,DATE,1,NaN,0
4,4,holder_name,TEXT,1,NaN,0
5,5,holder_type,TEXT,0,NaN,0
6,6,holder_type_raw,TEXT,0,NaN,0
7,7,hold_shares,REAL,0,NaN,0
8,8,hold_ratio_total,REAL,0,NaN,0
9,9,hold_ratio_float,REAL,0,NaN,0


获取记录数：1960
覆盖股票数：49


,id,stock_code,stock_name,report_date,holder_name,holder_type,holder_type_raw,hold_shares,hold_ratio_total,hold_ratio_float,change_status,change_shares,change_ratio,rank,is_float_holder,announce_date,data_source,created_at
0,5921,000001,None,2024-12-31,中国平安保险(集团)股份有限公司-集团本级-自有资金,NaN,None,9.618540e+09,49.56,NaN,不变,None,None,1,0,None,akshare,2026-08-19 11:59:50
1,5922,000001,None,2024-12-31,中国平安人寿保险股份有限公司-自有资金,NaN,None,1.186100e+09,6.11,NaN,不变,None,None,2,0,None,akshare,2026-08-19 11:59:50
2,5923,000001,None,2024-12-31,香港中央结算有限公司,NaN,None,7.468817e+08,3.85,NaN,减持,None,None,3,0,None,akshare,2026-08-19 11:59:50
3,5924,000001,None,2024-12-31,中国平安人寿保险股份有限公司-传统-普通保险产品,NaN,None,4.404787e+08,2.27,NaN,不变,None,None,4,0,None,akshare,2026-08-19 11:59:50
4,5925,000001,None,2024-12-31,中国证券金融股份有限公司,NaN,None,4.292327e+08,2.21,NaN,不变,None,None,5,0,None,akshare,2026-08-19 11:59:50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1955,5866,002304,None,2025-03-31,中国银行股份有限公司-易方达蓝筹精选混合型证券投资基金,其他,None,2.550000e+07,NaN,1.692767,减持,None,None,6,1,None,akshare,2026-08-18 12:52:55
1956,5867,002304,None,2025-03-31,香港中央结算有限公司,北向资金,None,2.315466e+07,NaN,1.537076,减持,None,None,7,1,None,akshare,2026-08-18 12:52:55
1957,5868,002304,None,2025-03-31,中国证券金融股份有限公司,证金公司,None,1.379004e+07,NaN,0.915425,不变,None,None,8,1,None,akshare,2026-08-18 12:52:55
1958,5869,002304,None,2025-03-31,邢福平,其他,None,1.029060e+07,NaN,0.683121,减持,None,None,9,1,None,akshare,2026-08-18 12:52:55


# 十大股东数据

## 验证数据库

In [36]:
# 查看十大股东数据是否已写入数据库
summary = pd.read_sql_query(
    """
    SELECT report_date, is_float_holder, COUNT(*) AS row_count,
           COUNT(DISTINCT stock_code) AS stock_count
    FROM top_holders
    GROUP BY report_date, is_float_holder
    ORDER BY report_date, is_float_holder
    """,
    connection,
)

display(summary)

print("总记录数:", pd.read_sql_query("SELECT COUNT(*) AS count FROM top_holders", connection).iloc[0, 0])

latest_records = pd.read_sql_query(
    """
    SELECT *
    FROM top_holders
    ORDER BY report_date DESC, stock_code, rank
    """,
    connection,
)

display(
    latest_records.groupby("report_date", group_keys=False)
    .head(5)
)

#display(pd.read_sql_query("SELECT * FROM top_holders LIMIT 5", connection))

,report_date,is_float_holder,row_count,stock_count
0,2024-12-31,0,8121,806
1,2024-12-31,1,8080,802
2,2025-03-31,0,8101,804
3,2025-03-31,1,8085,803
4,2025-06-30,0,8107,806
5,2025-06-30,1,8087,804
6,2025-09-30,0,8101,807
7,2025-09-30,1,8091,806
8,2025-12-31,0,8151,812
9,2025-12-31,1,8149,812


总记录数: 97344


,id,stock_code,stock_name,report_date,holder_name,holder_type,holder_type_raw,hold_shares,hold_ratio_total,hold_ratio_float,change_status,change_shares,change_ratio,rank,is_float_holder,announce_date,data_source,created_at
0,86021,000001,平安银行,2026-03-31,中国平安保险(集团)股份有限公司-集团本级-自有资金,NaN,None,9.618540e+09,49.56,NaN,不变,None,None,1,0,None,akshare,2026-08-20 06:09:10
1,94159,000001,平安银行,2026-03-31,中国平安保险(集团)股份有限公司-集团本级-自有资金,NaN,None,9.618540e+09,NaN,49.565795,不变,None,None,1,1,None,akshare,2026-08-20 06:22:51
2,86022,000001,平安银行,2026-03-31,中国平安人寿保险股份有限公司-自有资金,NaN,None,1.186100e+09,6.11,NaN,不变,None,None,2,0,None,akshare,2026-08-20 06:09:10
3,94160,000001,平安银行,2026-03-31,中国平安人寿保险股份有限公司-自有资金,NaN,None,1.186100e+09,NaN,6.112156,不变,None,None,2,1,None,akshare,2026-08-20 06:22:51
4,86023,000001,平安银行,2026-03-31,香港中央结算有限公司,NaN,None,5.707720e+08,2.94,NaN,减持,None,None,3,0,None,akshare,2026-08-20 06:09:10
16271,69719,000001,NaN,2025-12-31,中国平安保险(集团)股份有限公司-集团本级-自有资金,NaN,None,9.618540e+09,49.56,NaN,不变,None,None,1,0,None,akshare,2026-08-20 05:16:29
16272,77871,000001,NaN,2025-12-31,中国平安保险(集团)股份有限公司-集团本级-自有资金,NaN,None,9.618540e+09,NaN,49.565795,不变,None,None,1,1,None,akshare,2026-08-20 05:31:27
16273,69720,000001,NaN,2025-12-31,中国平安人寿保险股份有限公司-自有资金,NaN,None,1.186100e+09,6.11,NaN,不变,None,None,2,0,None,akshare,2026-08-20 05:16:29
16274,77872,000001,NaN,2025-12-31,中国平安人寿保险股份有限公司-自有资金,NaN,None,1.186100e+09,NaN,6.112156,不变,None,None,2,1,None,akshare,2026-08-20 05:31:27
16275,69721,000001,NaN,2025-12-31,香港中央结算有限公司,NaN,None,6.282911e+08,3.24,NaN,减持,None,None,3,0,None,akshare,2026-08-20 05:16:29


## stock_name为"None"

In [28]:
raw_df = fetch_top_holders_em("000001", "20250331", True)

print(raw_df.columns.tolist())
display(raw_df.head())

['名次', '股东名称', '股东性质', '股份类型', '持股数', '占总流通股本持股比例', '增减', '变动比率']


,名次,股东名称,股东性质,股份类型,持股数,占总流通股本持股比例,增减,变动比率
0,1,中国平安保险(集团)股份有限公司-集团本级-自有资金,保险公司,A股,9618540236,49.565869,不变,NaN
1,2,中国平安人寿保险股份有限公司-自有资金,保险公司,A股,1186100488,6.112165,不变,NaN
2,3,香港中央结算有限公司,其它,A股,658114653,3.391370,-88767070,-11.885024
3,4,中国平安人寿保险股份有限公司-传统-普通保险产品,保险产品,A股,440478714,2.269857,不变,NaN
4,5,中国证券金融股份有限公司,证券公司,A股,429232688,2.211904,不变,NaN


In [37]:
stocks = pd.read_sql_query(
    "SELECT * FROM stocks ORDER BY stock_code",
    connection,
)

stock_000001 = stocks[stocks["stock_code"] == "000001"]

print(f"stocks 表记录数：{len(stocks)}")
display(stock_000001)

stocks 表记录数：813


,stock_code,stock_name,total_shares,float_shares,industry,updated_at
0,000001,平安银行,None,None,None,2026-08-20 06:09:09


In [32]:
# 查看 index_components 表结构
index_components_schema = pd.read_sql_query(
    "PRAGMA table_info(index_components)",
    connection,
)
display(index_components_schema)

# 查看 index_components 数据量
component_stats = pd.read_sql_query(
    """
    SELECT
        index_code,
        effective_date,
        COUNT(*) AS row_count,
        COUNT(DISTINCT stock_code) AS stock_count
    FROM index_components
    GROUP BY index_code, effective_date
    ORDER BY index_code, effective_date
    """,
    connection,
)
display(component_stats)

# 查看指数成分股明细，并关联指数基本信息
component_details = pd.read_sql_query(
    """
    SELECT
        ic.index_code,
        i.index_name,
        i.exchange,
        ic.stock_code,
        ic.stock_name,
        ic.weight,
        ic.effective_date
    FROM index_components AS ic
    LEFT JOIN indices AS i
        ON i.index_code = ic.index_code
    ORDER BY ic.index_code, ic.effective_date, ic.stock_code
    """,
    connection,
)

print(f"成分股记录数：{len(component_details)}")
display(component_details.head(50))

# 随机查看每个指数的成分股
random_component_samples = pd.concat(
    [
        group.sample(n=min(10, len(group)), random_state=42)
        for _, group in component_details.groupby("index_code")
    ],
    ignore_index=True,
).sort_values(["index_code", "stock_code"])

display(random_component_samples)

,cid,name,type,notnull,dflt_value,pk
0,0,id,INTEGER,0,None,1
1,1,index_code,TEXT,1,None,0
2,2,stock_code,TEXT,1,None,0
3,3,stock_name,TEXT,0,None,0
4,4,weight,REAL,0,None,0
5,5,effective_date,DATE,0,None,0


,index_code,effective_date,row_count,stock_count
0,000300,2026-08-18,300,300
1,000300,2026-08-19,300,300
2,000688,2026-08-19,50,50
3,000905,2026-08-19,500,500
4,399006,2026-08-19,100,100


成分股记录数：1250


,index_code,index_name,exchange,stock_code,stock_name,weight,effective_date
0,000300,沪深300,sh,000001,平安银行,0.433,2026-08-18
1,000300,沪深300,sh,000002,万科A,0.087,2026-08-18
2,000300,沪深300,sh,000063,中兴通讯,0.418,2026-08-18
3,000300,沪深300,sh,000100,TCL科技,0.378,2026-08-18
4,000300,沪深300,sh,000157,中联重科,0.145,2026-08-18
5,000300,沪深300,sh,000166,申万宏源,0.162,2026-08-18
6,000300,沪深300,sh,000301,东方盛虹,0.119,2026-08-18
7,000300,沪深300,sh,000333,美的集团,1.634,2026-08-18
8,000300,沪深300,sh,000338,潍柴动力,0.577,2026-08-18
9,000300,沪深300,sh,000408,藏格矿业,0.243,2026-08-18


,index_code,index_name,exchange,stock_code,stock_name,weight,effective_date
6,000300,沪深300,sh,000425,徐工机械,0.282,2026-08-18
9,000300,沪深300,sh,002304,洋河股份,0.099,2026-08-19
3,000300,沪深300,sh,300014,亿纬锂能,0.317,2026-08-18
8,000300,沪深300,sh,300015,爱尔眼科,0.162,2026-08-18
0,000300,沪深300,sh,302132,中航成飞,0.066,2026-08-18
1,000300,沪深300,sh,600023,浙能电力,0.078,2026-08-19
7,000300,沪深300,sh,600588,用友网络,0.087,2026-08-19
4,000300,沪深300,sh,600886,国投电力,0.141,2026-08-18
2,000300,沪深300,sh,601998,中信银行,0.159,2026-08-19
5,000300,沪深300,sh,688041,海光信息,0.988,2026-08-18


## 数据库回填

In [40]:
pd.read_sql_query(
    """
    SELECT COUNT(*) AS empty_count
    FROM top_holders
    WHERE stock_name IS NULL OR stock_name = ''
    """,
    connection,
)
# 待修复记录

,empty_count
0,81073


In [43]:
stocks = pd.read_sql_query(
    """
    SELECT stock_code, stock_name
    FROM stocks
    ORDER BY stock_code
    """,
    connection,
)

print(f"stocks 记录数：{len(stocks)}")
display(stocks.head())
# 确认stocks 表中有多少数据

stocks 记录数：813


,stock_code,stock_name
0,000001,平安银行
1,000002,万科A
2,000009,中国宝安
3,000021,深科技
4,000027,深圳能源


In [ ]:
connection.execute(
    """
    UPDATE top_holders
    SET stock_name = (
        SELECT s.stock_name
        FROM stocks AS s
        WHERE s.stock_code = top_holders.stock_code
    )
    WHERE (top_holders.stock_name IS NULL OR top_holders.stock_name = '')
      AND EXISTS (
          SELECT 1
          FROM stocks AS s
          WHERE s.stock_code = top_holders.stock_code
            AND s.stock_name IS NOT NULL
            AND s.stock_name != ''
      )
    """
)

connection.commit()
# 回填操作